In [1]:
import os

import matplotlib.pyplot as plt

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["JAX_ENABLE_X64"] = "true"

import numpy as onp
import numpy.typing as npt
import jax
import jax.numpy as jnp
from typing import List, Callable, Iterable, Tuple

from gridops_multidim import BSplineInterpolationAxis, BSplineInterpolationGrid
from gridops_multidim import set_up_grid_axis

from gridops_multidim import make_multi_indices_one_particle, arbitrary_dim_outer
from gridops_multidim import create_anterpolation_operator
from gridops_multidim import create_restriction_operator, create_prolongation_operator, \
    create_interaction_operator
from gridops_multidim import create_compute_U_oneplus, create_compute_U_and_f_oneplus

import sys

sys.path.append("/home/florian/PhD/work/code/msm_for_nn/")

from msmfornn.splines.nesting import compute_J_zeroplus


In [2]:
%matplotlib notebook

# Basic settings

In [3]:
# geometry
length = 10.0
ndim = 3

# MSM
max_gridlevel = 6

# splines
p = 6
order = p - 1

# particles
n_particles = 1000

In [4]:
J_zeroplus = compute_J_zeroplus(p)
J = jnp.concatenate((J_zeroplus[::-1][:-1], J_zeroplus))

# Create particle configuration

In [5]:
# TODO: change back to more particles and random charge
#  (few particles and hard-coded charges serve visualization purposes only)

rng = onp.random.default_rng(1632794)
pos = rng.uniform(0., length, size=(n_particles, ndim))
chg = rng.uniform(-1., 1., size=n_particles)
# chg = onp.array([-2, -2, 1, 1, 1])

# Construct grids

In [6]:
grid_axes_all_levels = [None]  # there is no grid at level zero
for l in range(1, max_gridlevel + 1):
    h = length / (2 ** (max_gridlevel - l))
    print(l, h)
    grid_axis = set_up_grid_axis(length=length, h=h, p=p, J_zeroplus=J_zeroplus, periodic=False)
    grid_axes_all_levels.append(grid_axis)
    
# For now, we are just replicating the same axis along all dimensions
# (i.e., same grid spacing, box size, and boundary conditions along all dimensions)
grids_all_levels = [(None, ) * ndim] + [BSplineInterpolationGrid((ga,) * ndim) for ga in grid_axes_all_levels[1:]]
grid_level_one = grids_all_levels[1]
grid_level_two = grids_all_levels[2]

1 0.3125
2 0.625
3 1.25
4 2.5
5 5.0
6 10.0


# Functions

## Anterpolation

In [7]:
grid_level_one.shape

(39, 39, 39)

In [8]:
@jax.jit
def evaluate_bspline_basis_multiparticle(pos):
    return grid_level_one.evaluate_bspline_basis_multiparticle(pos)


@jax.jit
def evaluate_bspline_basis_gradient_multiparticle(pos):
    return grid_level_one.evaluate_bspline_basis_gradient_multiparticle(pos)

In [9]:
jax.device_put(pos)
%timeit evaluate_bspline_basis_multiparticle(pos)

1.44 ms ± 109 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [10]:
jax.device_put(pos)
%timeit evaluate_bspline_basis_gradient_multiparticle(pos)

2.65 ms ± 413 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [11]:
vals, inds = evaluate_bspline_basis_multiparticle(pos)
grads, _ = evaluate_bspline_basis_gradient_multiparticle(pos)

In [12]:
@jax.jit
def combined(pos):
    vals, inds = evaluate_bspline_basis_multiparticle(pos)
    grads, _ = evaluate_bspline_basis_gradient_multiparticle(pos)
    return inds, vals, grads

In [13]:
jax.device_put(pos)
%timeit combined(pos)

3.07 ms ± 1.41 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [14]:
anterpolate_level_one = create_anterpolation_operator(grid=grid_level_one)

In [15]:
jitted_anterpolate = jax.jit(anterpolate_level_one)

In [16]:
jax.device_put(pos)
jax.device_put(chg)

%timeit jitted_anterpolate(pos, chg).block_until_ready()

3.36 ms ± 374 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [17]:
gridcharge_level_one = anterpolate_level_one(pos, chg)

In [18]:
gridcharge_level_one

Array([0., 0., 0., ..., 0., 0., 0.], dtype=float64)

In [19]:
flat_indices = jnp.arange(gridcharge_level_one.shape[0])
unraveled_indices = jnp.unravel_index(flat_indices, grid_level_one.shape)
grid_points = []
for i, axis in enumerate(grid_level_one.axes):
    points = axis.to_raw_indices(unraveled_indices[i]) * axis.h
    grid_points.append(points)

In [20]:
mask = onp.abs(gridcharge_level_one) >= 0.01

x, y, z, = grid_points

fig = plt.figure()
ax = fig.add_subplot(projection="3d")
ax.scatter(x[mask], y[mask], z[mask], c=gridcharge_level_one[mask], s=10)
ax.scatter(pos[:, 0], pos[:, 1], pos[:, 2], c=chg, s=100)

plt.show()

<IPython.core.display.Javascript object>

## Restriction

In [21]:
def get_neigbhor_inds_on_fine_axis(
    idx_target: int,
    axis_source_fine: BSplineInterpolationAxis,
    axis_target_coarse: BSplineInterpolationAxis,
) -> jax.Array:
    """Get a target-grid index's neighbor indices on source grid."""
    raw_idx_targetgrid = axis_target_coarse.to_raw_indices(idx_target)
    raw_neighbor_inds_sourcegrid = 2 * raw_idx_targetgrid + jnp.arange(
        -p // 2, p // 2 + 1
    )
    neighbor_inds_sourcegrid = axis_source_fine.from_raw_indices(
        raw_neighbor_inds_sourcegrid
    )
    return neighbor_inds_sourcegrid

In [22]:
axis_level_1 = grid_axes_all_levels[1]
axis_level_2 = grid_axes_all_levels[2]

In [23]:
jnp.arange(axis_level_1.n_total)

Array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
       34, 35, 36, 37, 38], dtype=int64)

In [24]:
jnp.arange(axis_level_2.n_total)

Array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22], dtype=int64)

In [25]:
get_neigbhor_inds_on_fine_axis(0, axis_source_fine=axis_level_1, axis_target_coarse=axis_level_2)

Array([-6, -5, -4, -3, -2, -1,  0], dtype=int64)

In [26]:
from functools import partial

In [27]:
get_nb_inds_on_fa = partial(get_neigbhor_inds_on_fine_axis, axis_source_fine=grids_all_levels[1].axes[0], axis_target_coarse=grids_all_levels[2].axes[0])

In [28]:
get_nb_inds_on_fa = partial(get_neigbhor_inds_on_fine_axis, axis_source_fine=grids_all_levels[1].axes[1], axis_target_coarse=grids_all_levels[2].axes[1])

In [29]:
jax.vmap(get_nb_inds_on_fa)(jnp.arange(grids_all_levels[2].shape[0]))

Array([[-6, -5, -4, -3, -2, -1,  0],
       [-4, -3, -2, -1,  0,  1,  2],
       [-2, -1,  0,  1,  2,  3,  4],
       [ 0,  1,  2,  3,  4,  5,  6],
       [ 2,  3,  4,  5,  6,  7,  8],
       [ 4,  5,  6,  7,  8,  9, 10],
       [ 6,  7,  8,  9, 10, 11, 12],
       [ 8,  9, 10, 11, 12, 13, 14],
       [10, 11, 12, 13, 14, 15, 16],
       [12, 13, 14, 15, 16, 17, 18],
       [14, 15, 16, 17, 18, 19, 20],
       [16, 17, 18, 19, 20, 21, 22],
       [18, 19, 20, 21, 22, 23, 24],
       [20, 21, 22, 23, 24, 25, 26],
       [22, 23, 24, 25, 26, 27, 28],
       [24, 25, 26, 27, 28, 29, 30],
       [26, 27, 28, 29, 30, 31, 32],
       [28, 29, 30, 31, 32, 33, 34],
       [30, 31, 32, 33, 34, 35, 36],
       [32, 33, 34, 35, 36, 37, 38],
       [34, 35, 36, 37, 38, 39, 40],
       [36, 37, 38, 39, 40, 41, 42],
       [38, 39, 40, 41, 42, 43, 44]], dtype=int64)

In [30]:
boundary_condition_functions_source = [ga.wrap_or_invalidate_indices for ga in grid_level_one.axes]
neighbor_functions_individual_axes = []
for idx_cartesian in range(grid_level_one.ndim):
    nb_fun = partial(
        get_neigbhor_inds_on_fine_axis,
        axis_source_fine=grid_level_one.axes[idx_cartesian],
        axis_target_coarse=grid_level_two.axes[idx_cartesian],
    )
    neighbor_functions_individual_axes.append(nb_fun)
    
vmapped_neighbor_functions_individual_axes = [jax.vmap(nb_fun) for nb_fun in neighbor_functions_individual_axes]
    
indices_target_individual_axes = [jnp.arange(s) for s in grid_level_two.shape]

neighbor_inds_sourcegrid_individual_axes = [
    boundary_fun(vmapped_nb_fun(inds))
    for boundary_fun, vmapped_nb_fun, inds in zip(
        boundary_condition_functions_source,
        vmapped_neighbor_functions_individual_axes,
        indices_target_individual_axes,
    )
]

In [31]:
neighbor_inds_sourcegrid_individual_axes

[Array([[39, 39, 39, 39, 39, 39,  0],
        [39, 39, 39, 39,  0,  1,  2],
        [39, 39,  0,  1,  2,  3,  4],
        [ 0,  1,  2,  3,  4,  5,  6],
        [ 2,  3,  4,  5,  6,  7,  8],
        [ 4,  5,  6,  7,  8,  9, 10],
        [ 6,  7,  8,  9, 10, 11, 12],
        [ 8,  9, 10, 11, 12, 13, 14],
        [10, 11, 12, 13, 14, 15, 16],
        [12, 13, 14, 15, 16, 17, 18],
        [14, 15, 16, 17, 18, 19, 20],
        [16, 17, 18, 19, 20, 21, 22],
        [18, 19, 20, 21, 22, 23, 24],
        [20, 21, 22, 23, 24, 25, 26],
        [22, 23, 24, 25, 26, 27, 28],
        [24, 25, 26, 27, 28, 29, 30],
        [26, 27, 28, 29, 30, 31, 32],
        [28, 29, 30, 31, 32, 33, 34],
        [30, 31, 32, 33, 34, 35, 36],
        [32, 33, 34, 35, 36, 37, 38],
        [34, 35, 36, 37, 38, 39, 39],
        [36, 37, 38, 39, 39, 39, 39],
        [38, 39, 39, 39, 39, 39, 39]], dtype=int64),
 Array([[39, 39, 39, 39, 39, 39,  0],
        [39, 39, 39, 39,  0,  1,  2],
        [39, 39,  0,  1,  2,  3,  4

In [32]:
jax.vmap(make_multi_indices_one_particle)(*neighbor_inds_sourcegrid_individual_axes).shape

(23, 343, 3)

In [33]:
neighbor_inds_x_0 = neighbor_inds_sourcegrid_individual_axes[0][0]
neighbor_inds_y_0 = neighbor_inds_sourcegrid_individual_axes[1][0]
neighbor_inds_z_0 = neighbor_inds_sourcegrid_individual_axes[2][0]

neighbor_multi_indices_one_particle = make_multi_indices_one_particle(neighbor_inds_x_0, neighbor_inds_y_0, neighbor_inds_z_0)

In [34]:
neighbor_multi_indices_one_particle.shape

(343, 3)

In [35]:
# is_out_of_bounds = jnp.greater_equal(neighbor_multi_indices_one_particle, grid_level_one.shape[0])

In [36]:
# shape = jnp.array(grid_level_one.shape)
# is_not_periodic = ~jnp.array([ga.periodic for ga in grid_level_one.axes])
# # TODO: technically, what I need to check here is only < 0
# is_out_of_bounds = jnp.logical_or(neighbor_multi_indices_one_particle <0, neighbor_multi_indices_one_particle >= shape)
# is_out_of_bounds = is_out_of_bounds & is_not_periodic

In [37]:
[nb for nb in neighbor_functions_individual_axes]

[functools.partial(<function get_neigbhor_inds_on_fine_axis at 0x7f28d1fdbaf0>, axis_source_fine=BSplineInterpolationAxis(length=10.0, h=0.3125, p=6, J_zeroplus=array([0.625  , 0.46875, 0.1875 , 0.03125]), periodic=False, n_domain=33, n_total=39, to_raw_indices=<function set_up_grid_axis.<locals>.to_raw_indices at 0x7f28c73b4d30>, from_raw_indices=<function set_up_grid_axis.<locals>.from_raw_indices at 0x7f29304c7040>, wrap_indices_if_periodic=<function set_up_grid_axis.<locals>.wrap_indices_if_periodic at 0x7f29304c70d0>, wrap_or_invalidate_indices=<function set_up_grid_axis.<locals>.wrap_or_invalidate_indices at 0x7f29304c7310>, evaluate_bspline_basis_for_one_particle=<function set_up_grid_axis.<locals>.evaluate_bspline_basis_for_one_particle at 0x7f29304c71f0>, evaluate_bspline_basis_multi=<function set_up_grid_axis.<locals>.evaluate_bspline_basis_multi at 0x7f29304c73a0>, evaluate_bspline_basis_gradient_multi=<function set_up_grid_axis.<locals>.evaluate_bspline_basis_gradient_multi

In [38]:
def make_ravel_multi_inds_and_apply_bcs(grid: BSplineInterpolationGrid):
    shape = jnp.array(grid.shape)
    is_not_periodic = ~jnp.array([ga.periodic for ga in grid.axes])
    intentionally_out_of_bounds_index = grid.size
    
    def ravel_multi_inds_and_apply_bcs(multi_indices: jax.Array) -> jax.Array:
        # This handles periodic axes on its own due to the "wrap" keyword        
        flat_inds = jax.vmap(
            lambda multi_index: jnp.ravel_multi_index(
                multi_index, dims=shape, mode="wrap"
            )
        )(multi_indices)
        # Explicitly handle non-periodic axes
        is_out_of_bounds = jnp.logical_or(multi_indices < 0, multi_indices >= shape)
        is_out_of_bounds = (is_out_of_bounds & is_not_periodic).any(axis=1)
        flat_inds = jnp.where(is_out_of_bounds.ravel(), intentionally_out_of_bounds_index, flat_inds)

        return flat_inds

    return ravel_multi_inds_and_apply_bcs

In [39]:
ravel_multi_inds_and_apply_bcs = make_ravel_multi_inds_and_apply_bcs(grid_level_one)

In [40]:
neighbor_multi_indices_one_particle.shape

(343, 3)

In [41]:
grid_level_two.shape

(23, 23, 23)

In [42]:
neighbor_inds_sourcegrid_individual_axes

[Array([[39, 39, 39, 39, 39, 39,  0],
        [39, 39, 39, 39,  0,  1,  2],
        [39, 39,  0,  1,  2,  3,  4],
        [ 0,  1,  2,  3,  4,  5,  6],
        [ 2,  3,  4,  5,  6,  7,  8],
        [ 4,  5,  6,  7,  8,  9, 10],
        [ 6,  7,  8,  9, 10, 11, 12],
        [ 8,  9, 10, 11, 12, 13, 14],
        [10, 11, 12, 13, 14, 15, 16],
        [12, 13, 14, 15, 16, 17, 18],
        [14, 15, 16, 17, 18, 19, 20],
        [16, 17, 18, 19, 20, 21, 22],
        [18, 19, 20, 21, 22, 23, 24],
        [20, 21, 22, 23, 24, 25, 26],
        [22, 23, 24, 25, 26, 27, 28],
        [24, 25, 26, 27, 28, 29, 30],
        [26, 27, 28, 29, 30, 31, 32],
        [28, 29, 30, 31, 32, 33, 34],
        [30, 31, 32, 33, 34, 35, 36],
        [32, 33, 34, 35, 36, 37, 38],
        [34, 35, 36, 37, 38, 39, 39],
        [36, 37, 38, 39, 39, 39, 39],
        [38, 39, 39, 39, 39, 39, 39]], dtype=int64),
 Array([[39, 39, 39, 39, 39, 39,  0],
        [39, 39, 39, 39,  0,  1,  2],
        [39, 39,  0,  1,  2,  3,  4

In [43]:
i_x = 0
i_y = 1
i_z = 2

idx_gridpoint = 5

neighbor_multi_indices_one_particle = make_multi_indices_one_particle(
    neighbor_inds_sourcegrid_individual_axes[i_x][idx_gridpoint],
    neighbor_inds_sourcegrid_individual_axes[i_y][idx_gridpoint],
    neighbor_inds_sourcegrid_individual_axes[i_z][idx_gridpoint],
)

processed_flat_inds = ravel_multi_inds_and_apply_bcs(neighbor_multi_indices_one_particle)
processed_flat_inds

Array([ 6244,  6245,  6246,  6247,  6248,  6249,  6250,  6283,  6284,
        6285,  6286,  6287,  6288,  6289,  6322,  6323,  6324,  6325,
        6326,  6327,  6328,  6361,  6362,  6363,  6364,  6365,  6366,
        6367,  6400,  6401,  6402,  6403,  6404,  6405,  6406,  6439,
        6440,  6441,  6442,  6443,  6444,  6445,  6478,  6479,  6480,
        6481,  6482,  6483,  6484,  7765,  7766,  7767,  7768,  7769,
        7770,  7771,  7804,  7805,  7806,  7807,  7808,  7809,  7810,
        7843,  7844,  7845,  7846,  7847,  7848,  7849,  7882,  7883,
        7884,  7885,  7886,  7887,  7888,  7921,  7922,  7923,  7924,
        7925,  7926,  7927,  7960,  7961,  7962,  7963,  7964,  7965,
        7966,  7999,  8000,  8001,  8002,  8003,  8004,  8005,  9286,
        9287,  9288,  9289,  9290,  9291,  9292,  9325,  9326,  9327,
        9328,  9329,  9330,  9331,  9364,  9365,  9366,  9367,  9368,
        9369,  9370,  9403,  9404,  9405,  9406,  9407,  9408,  9409,
        9442,  9443,

In [44]:
gridcharge_level_one.shape

(59319,)

In [45]:
gridcharge_level_one[processed_flat_inds]

Array([-2.73857463e-02, -6.50608286e-02, -5.41562193e-02, -6.35773519e-03,
        1.00264575e-02,  5.19595737e-02,  3.98167088e-02, -1.46760207e-02,
       -1.10194457e-01, -1.08191132e-01, -1.85184582e-02, -2.49010353e-05,
        2.79730227e-03,  2.14371308e-03,  2.60634716e-02, -2.10923690e-02,
       -9.10461110e-02, -4.55084574e-02, -3.29359575e-03, -1.04913551e-06,
        2.47392737e-08,  2.36544543e-02,  5.07005334e-03, -5.73488269e-02,
       -3.74163488e-02, -2.90222959e-03, -9.55093161e-07,  0.00000000e+00,
        2.36190420e-03,  2.72201041e-04, -6.89809745e-03, -4.50704267e-03,
       -3.50199224e-04, -1.15253285e-07,  0.00000000e+00,  2.78745084e-06,
       -1.96831226e-06, -1.78203810e-05, -1.14508805e-05, -8.89695529e-07,
       -2.92805710e-10, -4.97319097e-14,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
       -3.10346366e-11, -2.57857127e-03, -1.25951685e-02, -1.10067027e-02,
       -1.20647680e-03,  

In [46]:
multi_inds_target = make_multi_indices_one_particle(*[jnp.arange(s) for s in grid_level_two.shape])
flat_inds_target = make_ravel_multi_inds_and_apply_bcs(grid_level_two)(multi_inds_target)

In [47]:
multi_inds_target.shape, flat_inds_target.shape

((12167, 3), (12167,))

In [48]:
neighbor_inds_source_individual_axes = [nb_fun(idx) for nb_fun, idx in zip(neighbor_functions_individual_axes, multi_inds_target[100])]
make_multi_indices_one_particle(*neighbor_inds_source_individual_axes).shape

(343, 3)

In [49]:
def get_neighbor_multi_inds_one_gridpoint(multi_idx_target):
    neighbor_inds_source_individual_axes = [nb_fun(idx) for nb_fun, idx in zip(neighbor_functions_individual_axes, multi_idx_target)]
    return make_multi_indices_one_particle(*neighbor_inds_source_individual_axes)

In [50]:
def get_neighbor_flat_inds_one_gridpoint(multi_idx_target):
    neighbor_inds_source_individual_axes = [nb_fun(idx) for nb_fun, idx in zip(neighbor_functions_individual_axes, multi_idx_target)]
    multi_inds_source = make_multi_indices_one_particle(*neighbor_inds_source_individual_axes)
    return ravel_multi_inds_and_apply_bcs(multi_inds_source)

In [51]:
get_neighbor_multi_inds_one_gridpoint(multi_inds_target[0]).shape

(343, 3)

In [52]:
jax.vmap(get_neighbor_multi_inds_one_gridpoint)(multi_inds_target).shape

(12167, 343, 3)

In [53]:
get_neighbor_flat_inds_one_gridpoint(multi_inds_target[100])

Array([59319, 59319, 59319, 59319, 59319, 59319, 59319, 59319, 59319,
       59319, 59319, 59319, 59319, 59319, 59319, 59319, 59319, 59319,
       59319, 59319, 59319, 59319, 59319, 59319, 59319, 59319, 59319,
       59319, 59319, 59319, 59319, 59319, 59319, 59319, 59319, 59319,
       59319, 59319, 59319, 59319, 59319, 59319, 59319, 59319, 59319,
       59319, 59319, 59319, 59319, 59319, 59319, 59319, 59319, 59319,
       59319, 59319, 59319, 59319, 59319, 59319, 59319, 59319, 59319,
       59319, 59319, 59319, 59319, 59319, 59319, 59319, 59319, 59319,
       59319, 59319, 59319, 59319, 59319, 59319, 59319, 59319, 59319,
       59319, 59319, 59319, 59319, 59319, 59319, 59319, 59319, 59319,
       59319, 59319, 59319, 59319, 59319, 59319, 59319, 59319, 59319,
       59319, 59319, 59319, 59319, 59319, 59319, 59319, 59319, 59319,
       59319, 59319, 59319, 59319, 59319, 59319, 59319, 59319, 59319,
       59319, 59319, 59319, 59319, 59319, 59319, 59319, 59319, 59319,
       59319, 59319,

In [54]:
flat_neighbor_inds_all_target_gridpoints = jax.vmap(get_neighbor_flat_inds_one_gridpoint)(multi_inds_target)
J_multidim_flat = arbitrary_dim_outer(*([J] * grid_level_one.ndim)).ravel()

In [55]:
(flat_neighbor_inds_all_target_gridpoints * J_multidim_flat).shape

(12167, 343)

In [56]:
(flat_neighbor_inds_all_target_gridpoints * J_multidim_flat).sum(axis=1).shape

(12167,)

In [57]:
neighbor_multi_inds_all_target_points = jax.vmap(make_multi_indices_one_particle)(*neighbor_inds_sourcegrid_individual_axes)

In [58]:
neighbor_multi_inds_all_target_points.shape

(23, 343, 3)

In [59]:
(neighbor_multi_indices_one_particle == 0).sum() + (neighbor_multi_indices_one_particle == 15).sum()

Array(0, dtype=int64)

In [60]:
neighbor_multi_indices_one_particle.size

1029

In [61]:
neighbor_inds_sourcegrid = jax.vmap(make_multi_indices_one_particle)(*neighbor_inds_sourcegrid_individual_axes)
neighbor_inds_sourcegrid.shape

(23, 343, 3)

In [62]:
arr_on_source_grid = jnp.zeros(grid_level_one.shape)

In [63]:
arr_on_source_grid[neighbor_inds_sourcegrid[0]].shape

(343, 3, 39, 39)

In [64]:
neighbor_inds_sourcegrid[0]

Array([[39, 39, 39],
       [39, 39, 39],
       [39, 39, 39],
       ...,
       [ 0,  0, 39],
       [ 0,  0, 39],
       [ 0,  0,  0]], dtype=int64)

In [65]:
arr_on_source_grid[(0, 1, 2)]

Array(0., dtype=float64)

In [66]:
arr = jnp.outer(jnp.arange(1, 4), jnp.arange(1, 4))

In [67]:
arr

Array([[1, 2, 3],
       [2, 4, 6],
       [3, 6, 9]], dtype=int64)

In [68]:
@jax.jit
def myfun(x, inds):
    return 2 * x.at[inds].get(mode="fill", fill_value=0.0)

In [69]:
myfun(arr, inds=(0, 2))

Array(6, dtype=int64)

In [70]:
big_arr = onp.full((6, 6), -1)
for i in range(big_arr.shape[0]):
    for j in range(big_arr.shape[1]):
        big_arr[i, j] = myfun(arr, inds=(i, j))

In [71]:
jnp.asarray(big_arr)

Array([[ 2,  4,  6,  0,  0,  0],
       [ 4,  8, 12,  0,  0,  0],
       [ 6, 12, 18,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0]], dtype=int64)

In [72]:
restrict_1to2 = create_restriction_operator(grid_source_fine=grid_level_one, grid_target_coarse=grid_level_two)

In [73]:
gridcharge_level_two_from_restriction = restrict_1to2(gridcharge_level_one)

In [74]:
anterpolate_level_two = create_anterpolation_operator(grid_level_two)
gridcharge_level_two = anterpolate_level_two(pos, chg)

In [75]:
jnp.allclose(gridcharge_level_two, gridcharge_level_two_from_restriction)

Array(True, dtype=bool)

In [76]:
jitted_anterpolate_level_two = jax.jit(anterpolate_level_two)
jitted_restrict_1to2 = jax.jit(restrict_1to2)

In [77]:
jax.device_put(pos)
jax.device_put(chg)

%timeit jitted_anterpolate_level_two(pos, chg).block_until_ready()

3.24 ms ± 393 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [78]:
jax.device_put(gridcharge_level_one)

%timeit jitted_restrict_1to2(gridcharge_level_one).block_until_ready()

3.84 ms ± 405 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [79]:
from gridops import create_restriction_operator as create_restriction_operator_1d

In [80]:
i_x = 0
i_y = 1
i_z = 2

restrict_1to2_1d_x = create_restriction_operator_1d(
    grid_source_fine=grid_level_one.axes[i_x],
    grid_target_coarse=grid_level_two.axes[i_x],
)
restrict_1to2_1d_y = create_restriction_operator_1d(
    grid_source_fine=grid_level_one.axes[i_y],
    grid_target_coarse=grid_level_two.axes[i_y],
)
restrict_1to2_1d_z = create_restriction_operator_1d(
    grid_source_fine=grid_level_one.axes[i_z],
    grid_target_coarse=grid_level_two.axes[i_z],
)

restriction_funcs_1to2_1d_individual_axes = [
    restrict_1to2_1d_x, restrict_1to2_1d_y, restrict_1to2_1d_z
]

In [81]:
gridcharge_level_one_reshaped = gridcharge_level_one.reshape(grid_level_one.shape)

In [82]:
gridcharge_level_one_reshaped.shape

(39, 39, 39)

In [83]:
gridcharge_level_one_reshaped[:, 0, 0].shape

(39,)

In [84]:
def create_restriction_operator_alternative(
    grid_source_fine: BSplineInterpolationGrid,
    grid_target_coarse: BSplineInterpolationGrid,
) -> Callable:
    restriction_funcs_1d_individual_axes = []
    for axis_source, axis_target in zip(
        grid_source_fine.axes, grid_target_coarse.axes
    ):
        restriction_funcs_1d_individual_axes.append(
            create_restriction_operator_1d(
                grid_source_fine=axis_source,
                grid_target_coarse=axis_target,
            )
        )

    def restrict(in_array_fine):
        out_array_coarse = in_array_fine

        for idx_ax, rf_1d in enumerate(
            restriction_funcs_1d_individual_axes
        ):
            out_array_coarse = jnp.apply_along_axis(
                func1d=rf_1d,
                axis=idx_ax,
                arr=out_array_coarse,
            )

        return out_array_coarse

    return restrict

In [86]:
restrict_1to2_alternative = create_restriction_operator_alternative(grid_source_fine=grid_level_one, grid_target_coarse=grid_level_two)

In [87]:
gridcharge_level_two_alternative = restrict_1to2_alternative(gridcharge_level_one_reshaped)

In [88]:
jnp.allclose(gridcharge_level_two, gridcharge_level_two_alternative.ravel())

Array(True, dtype=bool)

In [92]:
jax.device_put(pos)
jax.device_put(chg)

%timeit jitted_anterpolate_level_two(pos, chg).block_until_ready()

4.81 ms ± 521 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [93]:
jax.device_put(gridcharge_level_one)

%timeit jitted_restrict_1to2(gridcharge_level_one).block_until_ready()

4.81 ms ± 102 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [94]:
jax.device_put(gridcharge_level_one_reshaped)

%timeit restrict_1to2_alternative(gridcharge_level_one_reshaped).block_until_ready()

154 ms ± 8.12 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
